# Isolation Forest Anomaly Detection - Model Training

This notebook trains and evaluates an Isolation Forest model for detecting
anomalous (potentially fraudulent) transactions in the RiskPulse pipeline.

## Steps
1. Generate synthetic transaction data with fraud patterns
2. Feature engineering and exploration
3. Time-based train/val/test split
4. Model training with hyperparameter tuning
5. Evaluation and threshold analysis
6. Model serialization

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
)

# Add project root
PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.fraud_detection.anomaly_detector import AnomalyDetector, ANOMALY_FEATURES
from ml.training.train_anomaly_detector import (
    generate_synthetic_data,
    time_based_split,
    cross_validate_timeseries,
)
from ml.evaluation.evaluate_model import ModelEvaluator

RANDOM_STATE = 42
print(f"Project root: {PROJECT_ROOT}")

## 1. Generate Synthetic Data

In [ ]:
X, y = generate_synthetic_data(n_samples=10000, fraud_ratio=0.02, random_state=RANDOM_STATE)

print(f"Dataset shape: {X.shape}")
print(f"Fraud samples: {y.sum():.0f} ({y.mean()*100:.2f}%)")
print(f"Legitimate samples: {(y==0).sum():.0f}")
print(f"\nFeature columns: {ANOMALY_FEATURES}")
X[ANOMALY_FEATURES].describe()

## 2. Feature Distribution Analysis

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 12))
axes = axes.ravel()

for i, feat in enumerate(ANOMALY_FEATURES):
    ax = axes[i]
    ax.hist(X[feat][y == 0], bins=50, alpha=0.6, label="Legitimate", density=True)
    ax.hist(X[feat][y == 1], bins=50, alpha=0.6, label="Fraud", density=True)
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.suptitle("Feature Distributions: Legitimate vs Fraud", y=1.02, fontsize=14)
plt.show()

## 3. Time-Based Train/Val/Test Split

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = time_based_split(X, y)

print(f"Train: {len(X_train)} samples ({y_train.mean()*100:.2f}% fraud)")
print(f"Val:   {len(X_val)} samples ({y_val.mean()*100:.2f}% fraud)")
print(f"Test:  {len(X_test)} samples ({y_test.mean()*100:.2f}% fraud)")

## 4. Cross-Validation

In [ ]:
cv_metrics = cross_validate_timeseries(
    X_train, y_train,
    n_estimators=200,
    max_samples="auto",
    contamination=0.02,
    max_features=0.8,
    n_splits=5,
    random_state=RANDOM_STATE,
)

print("Cross-Validation Results:")
for metric, scores in cv_metrics.items():
    print(f"  {metric}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

## 5. Train Final Model

In [ ]:
detector = AnomalyDetector(
    n_estimators=200,
    max_samples="auto",
    contamination=0.02,
    max_features=0.8,
    random_state=RANDOM_STATE,
)

detector.fit(X_train)
print(f"Model version: {detector.model_version}")
print(f"Is fitted: {detector.is_fitted}")

## 6. Evaluate on Test Set

In [ ]:
evaluator = ModelEvaluator(detector)
metrics = evaluator.evaluate(X_test, y_test)

print("Test Set Metrics:")
print(f"  Precision:    {metrics.precision:.4f}")
print(f"  Recall:       {metrics.recall:.4f}")
print(f"  F1 Score:     {metrics.f1_score:.4f}")
print(f"  AUC-ROC:      {metrics.auc_roc:.4f}")
print(f"  AUC-PR:       {metrics.auc_pr:.4f}")
print(f"  FPR:          {metrics.false_positive_rate:.4f}")
print(f"\nConfusion Matrix: {metrics.confusion}")
print(f"\nPasses thresholds: {metrics.passes_thresholds()}")

## 7. ROC and Precision-Recall Curves

In [ ]:
scores = detector.get_anomaly_scores(X_test)
neg_scores = -scores  # Higher = more anomalous

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, neg_scores)
auc_val = roc_auc_score(y_test, neg_scores)
ax1.plot(fpr, tpr, label=f"AUC = {auc_val:.4f}")
ax1.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.set_title("ROC Curve")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, neg_scores)
ax2.plot(rec, prec)
ax2.axhline(y=0.8, color="r", linestyle="--", alpha=0.5, label="Min Precision (0.8)")
ax2.axvline(x=0.85, color="g", linestyle="--", alpha=0.5, label="Min Recall (0.85)")
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall Curve")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Threshold Analysis

In [ ]:
threshold_results = evaluator.threshold_analysis(X_test, y_test)

print(f"Optimal threshold: {threshold_results.optimal_threshold:.4f}")
print(f"Optimal F1: {threshold_results.optimal_f1:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(threshold_results.thresholds, threshold_results.precisions, label="Precision")
ax.plot(threshold_results.thresholds, threshold_results.recalls, label="Recall")
ax.plot(threshold_results.thresholds, threshold_results.f1_scores, label="F1")
ax.plot(threshold_results.thresholds, threshold_results.fprs, label="FPR", linestyle="--")
ax.axvline(x=threshold_results.optimal_threshold, color="k", linestyle=":", label="Optimal")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Metrics vs Decision Threshold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Latency Benchmark

In [ ]:
latency = evaluator.benchmark_latency(X_test, n_iterations=200)

print(f"Mean latency:   {latency.mean_ms:.3f} ms")
print(f"Median latency: {latency.median_ms:.3f} ms")
print(f"P95 latency:    {latency.p95_ms:.3f} ms")
print(f"P99 latency:    {latency.p99_ms:.3f} ms")
print(f"Max latency:    {latency.max_ms:.3f} ms")
print(f"Meets SLA (<10ms): {latency.meets_sla(10.0)}")

## 10. Save Model

In [ ]:
model_dir = PROJECT_ROOT / "ml" / "models" / "isolation_forest"
save_path = detector.save(model_dir)
print(f"Model saved to: {save_path}")

# Verify load
loaded = AnomalyDetector.load(model_dir)
print(f"Model loaded: version={loaded.model_version}, fitted={loaded.is_fitted}")

# Verify predictions match
sample = X_test.iloc[0].to_dict()
orig_result = detector.predict(sample)
loaded_result = loaded.predict(sample)
print(f"\nOriginal score:  {orig_result.anomaly_score:.6f}")
print(f"Loaded score:    {loaded_result.anomaly_score:.6f}")
print(f"Scores match:    {abs(orig_result.anomaly_score - loaded_result.anomaly_score) < 1e-10}")

## 11. Generate Full Report

In [ ]:
report = evaluator.generate_report(
    X_test, y_test,
    output_path=model_dir / "evaluation_report.json",
)

print("Production Readiness:")
for check, passed in report["production_readiness"].items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check}")